<a href="https://colab.research.google.com/github/WeegorMartins/customer-decisioning-lab/blob/main/notebooks/04_card_multitreatment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip -q install lightgbm

In [11]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier

from google.colab import drive
drive.mount("/content/drive")

SEED = 42

PROJECT_DIR = Path(
    "/content/drive/MyDrive/customer-decisioning-lab"
)

CARD_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "card_experiment_fast.parquet"
)

cards = pd.read_parquet(CARD_PATH)

print(cards.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(80000, 21)


In [12]:
train_cards = cards[
    cards["decision_date"] < "2026-01-01"
].copy()

valid_cards = cards[
    cards["decision_date"].between(
        "2026-01-01",
        "2026-02-28"
    )
].copy()

test_cards = cards[
    cards["decision_date"] >= "2026-03-01"
].copy()

print("Treino:", train_cards.shape)
print("Validação:", valid_cards.shape)
print("Teste OOT:", test_cards.shape)

Treino: (53315, 21)
Validação: (8938, 21)
Teste OOT: (17747, 21)


In [13]:
NUMERIC_FEATURES = [
    "tenure_months",
    "age",
    "spend_90d",
    "transactions_90d",
    "recency_days",
    "digital_sessions_30d",
    "merchant_diversity_90d",
    "contacts_30d",
    "complaints_180d",
    "marketing_consent"
]

SEGMENTS = [
    "novo",
    "ativo",
    "em_risco",
    "inativo",
    "ocasional"
]

def build_features(data):
    result = data[
        NUMERIC_FEATURES
    ].copy()

    for segment in SEGMENTS:
        result[
            f"segment_{segment}"
        ] = (
            data["lifecycle_segment"]
            == segment
        ).astype(int)

    return result

X_train_cards = build_features(
    train_cards
)
X_valid_cards = build_features(
    valid_cards
)
X_test_cards = build_features(
    test_cards
)

FEATURE_COLUMNS = X_train_cards.columns.tolist()

print(FEATURE_COLUMNS)

['tenure_months', 'age', 'spend_90d', 'transactions_90d', 'recency_days', 'digital_sessions_30d', 'merchant_diversity_90d', 'contacts_30d', 'complaints_180d', 'marketing_consent', 'segment_novo', 'segment_ativo', 'segment_em_risco', 'segment_inativo', 'segment_ocasional']


In [14]:
CARD_MODEL_PARAMS = {
    "n_estimators": 250,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_child_samples": 100,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1
}

def fit_action_models(
    data,
    feature_matrix,
    action
):
    action_mask = data[
        "treatment"
    ].eq(action).to_numpy()

    control_mask = data[
        "treatment"
    ].eq(0).to_numpy()

    action_model = LGBMClassifier(
        **CARD_MODEL_PARAMS
    )

    control_model = LGBMClassifier(
        **CARD_MODEL_PARAMS
    )

    action_model.fit(
        feature_matrix.loc[action_mask],
        data.loc[
            action_mask,
            "converted_90d"
        ]
    )

    control_model.fit(
        feature_matrix.loc[control_mask],
        data.loc[
            control_mask,
            "converted_90d"
        ]
    )

    return action_model, control_model

In [15]:
action_models = {}

for action in [1, 2, 3]:
    print("Treinando ação:", action)

    action_models[action] = (
        fit_action_models(
            train_cards.reset_index(
                drop=True
            ),
            X_train_cards.reset_index(
                drop=True
            ),
            action
        )
    )

print("TODAS AS AÇÕES FORAM TREINADAS")

Treinando ação: 1
Treinando ação: 2
Treinando ação: 3
TODAS AS AÇÕES FORAM TREINADAS


In [16]:
scored_cards = test_cards.reset_index(
    drop=True
).copy()

X_score = X_test_cards.reset_index(
    drop=True
)

for action, models in action_models.items():
    action_model, control_model = models

    p_action = action_model.predict_proba(
        X_score
    )[:, 1]

    p_control = control_model.predict_proba(
        X_score
    )[:, 1]

    scored_cards[
        f"p_action_{action}"
    ] = p_action

    scored_cards[
        f"p_control_for_{action}"
    ] = p_control

    scored_cards[
        f"uplift_action_{action}"
    ] = p_action - p_control

display(
    scored_cards[[
        "customer_id",
        "uplift_action_1",
        "uplift_action_2",
        "uplift_action_3"
    ]].head()
)

,customer_id,uplift_action_1,uplift_action_2,uplift_action_3
0,16,0.127066,0.007116,0.024043
1,25,0.326859,0.186927,0.018298
2,28,-0.020698,0.031452,-0.003052
3,38,0.075112,0.037500,0.016931
4,50,0.064241,0.061722,0.180416


In [17]:
uplift_columns = [
    "uplift_action_1",
    "uplift_action_2",
    "uplift_action_3"
]

scored_cards["best_action_by_uplift"] = (
    scored_cards[uplift_columns]
    .idxmax(axis=1)
    .str.extract(r"(\d+)")
    .astype(int)
)

scored_cards["best_uplift"] = (
    scored_cards[uplift_columns]
    .max(axis=1)
)

scored_cards.loc[
    scored_cards["best_uplift"] <= 0,
    "best_action_by_uplift"
] = 0

display(
    scored_cards[
        "best_action_by_uplift"
    ].value_counts()
)

,count
best_action_by_uplift,
1,7812
2,5348
3,3700
0,887


In [18]:
SCORED_PATH = (
    PROJECT_DIR
    / "results"
    / "card_oot_scored.parquet"
)

SCORED_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

scored_cards.to_parquet(
    SCORED_PATH,
    index=False
)

MODELS_DIR = PROJECT_DIR / "models"
MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

for action, models in action_models.items():
    joblib.dump(
        models[0],
        MODELS_DIR
        / f"card_action_{action}.joblib"
    )
    joblib.dump(
        models[1],
        MODELS_DIR
        / f"card_control_for_{action}.joblib"
    )

print("SCORING MULTIAÇÃO SALVO")

SCORING MULTIAÇÃO SALVO
